# Metal Cutting from First Principles

**A step-by-step walk through one turning job**

Imagine a steel bar spinning in a lathe while a fixed cutting tool peels a chip off its surface. In this notebook we start from what the operator chooses (the bar size, the speed, the feed and the depth of cut) and work out, one step at a time, everything else: how fast the spindle turns, how long the cut takes, how much metal comes off, how hard the tool pushes, and whether the machine has enough power.

The formulas follow the Sandvik Coromant *Metal Cutting Training Handbook*. Every formula is just a line of Python, so you can change a number and run it again.

In [ ]:
import math

## Step 0. The job

We are turning a steel shaft. These are the numbers we choose:

In [ ]:
D  = 80      # diameter of the bar, mm
vc = 200     # cutting speed recommended for this tool and material, m/min
fn = 0.3     # feed: how far the tool moves along the bar per revolution, mm
ap = 3       # depth of cut: how deep the tool goes into the bar, mm
L  = 150     # length of the cut along the bar, mm

## Step 1. How fast does the surface move? Cutting speed

In one revolution a point on the surface travels once around the circumference, $\pi \cdot D$. If the bar turns $n$ times per minute, the surface moves $\pi \cdot D \cdot n$ millimetres per minute. Dividing by 1000 gives metres per minute:

$$v_c = \frac{\pi \cdot D \cdot n}{1000}$$

For example, at 500 rpm:

In [ ]:
n_example = 500
print("circumference:", round(math.pi * D, 1), "mm")
print("cutting speed at 500 rpm:", round(math.pi * D * n_example / 1000, 1), "m/min")

## Step 2. What rpm do we set? Spindle speed

In practice it works the other way round. The tool maker recommends a cutting speed $v_c$, and the machine needs an rpm. We rearrange the same formula:

$$n = \frac{v_c \cdot 1000}{\pi \cdot D}$$

In [ ]:
n = vc * 1000 / (math.pi * D)
print("spindle speed n =", round(n), "rpm")

Notice that a smaller diameter needs a higher rpm for the same cutting speed:

In [ ]:
for d in [200, 100, 50, 20, 10]:
    print(f"D = {d:3d} mm  ->  n = {vc * 1000 / (math.pi * d):6.0f} rpm")

## Step 3. How fast does the tool travel? Feed speed

Each revolution the tool moves $f_n$ mm along the bar. With $n$ revolutions per minute:

$$v_f = f_n \cdot n \quad \text{(mm/min)}$$

In [ ]:
vf = fn * n
print("feed speed vf =", round(vf), "mm/min")

## Step 4. How long does the cut take? Machining time

Time is distance divided by speed:

$$T_c = \frac{L}{v_f}$$

In [ ]:
Tc = L / vf
print("machining time =", round(Tc, 2), "min  =", round(Tc * 60), "seconds")

## Step 5. How much metal comes off? Metal removal rate

The chip has a cross section of $a_p \times f_n$ (depth times feed), and the surface passes the tool at $v_c$ metres per minute. Volume per minute is area times speed:

$$Q = v_c \cdot a_p \cdot f_n \quad \text{(cm}^3\text{/min)}$$

(The units work out neatly: m/min × mm × mm = 1000 mm³/min = 1 cm³/min.)

In [ ]:
Q = vc * ap * fn
print("metal removal rate Q =", round(Q), "cm^3/min")
print("that is about", round(Q * 7.85 / 1000, 2), "kg of steel per minute")

## Step 6. How thick is the chip?

The cutting edge is usually set at an angle to the bar, called the entering angle $\kappa_r$. The edge then spreads the feed over a longer length, and the chip comes off thinner than the feed:

$$h = f_n \cdot \sin \kappa_r$$

At $\kappa_r = 90°$ the chip is exactly as thick as the feed.

In [ ]:
kr = 95   # entering angle, degrees
h = fn * math.sin(math.radians(kr))
print("chip thickness h =", round(h, 3), "mm")

## Step 7. How hard is the material to cut? Specific cutting force

Every material has a **specific cutting force** $k_c$: the force needed to cut a chip with a cross section of 1 mm². It depends on the material and, perhaps surprisingly, on the chip thickness. Thin chips cost more force per mm²:

$$k_c = k_{c1} \cdot h^{-m_c}$$

$k_{c1}$ and $m_c$ are material constants taken from the handbook's material table. For an ordinary steel, roughly $k_{c1} = 1500$ N/mm² and $m_c = 0.25$.

In [ ]:
kc1 = 1500   # N/mm^2, specific cutting force at 1 mm chip thickness
mc  = 0.25   # how strongly kc rises for thinner chips

kc = kc1 * h ** (-mc)
print("specific cutting force kc =", round(kc), "N/mm^2")

In [ ]:
# Thin chips need more force per mm^2
for hh in [1.0, 0.5, 0.3, 0.1, 0.05]:
    print(f"h = {hh:4.2f} mm  ->  kc = {kc1 * hh ** (-mc):5.0f} N/mm^2")

## Step 8. How hard does the tool push? Cutting force

Force = (force per mm²) × (chip area in mm²):

$$F_c = k_c \cdot a_p \cdot f_n$$

In [ ]:
Fc = kc * ap * fn
print("cutting force Fc =", round(Fc), "N   (about", round(Fc / 9.81), "kg weight)")

## Step 9. How much power does it take?

Power is force times speed. The force is $F_c$ newtons and the surface moves at $v_c$ m/min, which is $v_c/60$ m/s. Dividing by 1000 turns watts into kilowatts:

$$P_c = \frac{F_c \cdot v_c}{60 \cdot 1000}$$

Substituting $F_c = k_c \cdot a_p \cdot f_n$ gives the formula printed in the handbook:

$$P_c = \frac{v_c \cdot a_p \cdot f_n \cdot k_c}{60 \cdot 10^3} \quad \text{(kW)}$$

In [ ]:
Pc = vc * ap * fn * kc / (60 * 1000)
print("net cutting power Pc =", round(Pc, 2), "kW")

## Step 10. Can the machine do it?

The motor loses some power in the drive, so the spindle delivers only a fraction of its rating (the efficiency, often about 80 %).

In [ ]:
machine_power = 10    # kW, motor rating
efficiency    = 0.8

needed = Pc / efficiency
print("motor power needed:", round(needed, 1), "kW of", machine_power, "kW available")
print("OK" if needed <= machine_power else "Too much: reduce feed, depth or speed")

## Putting it all together

Here is the whole chain in one function. Change any input and see what happens to time and power.

In [ ]:
def turning_job(D, vc, fn, ap, L, kr=95, kc1=1500, mc=0.25):
    n  = vc * 1000 / (math.pi * D)
    vf = fn * n
    Tc = L / vf
    Q  = vc * ap * fn
    h  = fn * math.sin(math.radians(kr))
    kc = kc1 * h ** (-mc)
    Pc = vc * ap * fn * kc / 60000
    print(f"n = {n:5.0f} rpm | vf = {vf:5.0f} mm/min | time = {Tc*60:4.0f} s | "
          f"Q = {Q:4.0f} cm3/min | kc = {kc:4.0f} N/mm2 | Pc = {Pc:5.2f} kW")

turning_job(D=80, vc=200, fn=0.3, ap=3, L=150)

In [ ]:
# What if we double the feed?
turning_job(D=80, vc=200, fn=0.6, ap=3, L=150)

# What if we double the cutting speed instead?
turning_job(D=80, vc=400, fn=0.3, ap=3, L=150)

**Discussion.** Both changes halve the time. Why does doubling the feed take slightly *less* power than doubling the speed? (Hint: look at $k_c$ and Step 7.) And which one do you think wears the tool out faster?

## The same ideas in milling

In milling the tool spins instead of the workpiece, and it has several teeth. Each tooth takes a small bite $f_z$ (feed per tooth), so the table moves at

$$v_f = f_z \cdot z_c \cdot n$$

where $z_c$ is the number of teeth. The removal rate is width × depth × feed speed, and power follows the same logic as before:

$$Q = \frac{a_p \cdot a_e \cdot v_f}{1000} \qquad P_c = \frac{a_p \cdot a_e \cdot v_f \cdot k_c}{60 \cdot 10^6}$$

Here $a_e$ is the width of the cut. (To keep it simple we use the feed per tooth as the chip thickness; the handbook shows how to correct for narrow cuts.)

In [ ]:
D_cutter = 63    # mm
z        = 5     # teeth
vc_mill  = 250   # m/min
fz       = 0.15  # mm per tooth
ap_mill  = 2     # mm depth
ae       = 40    # mm width of cut

n_mill  = vc_mill * 1000 / (math.pi * D_cutter)
vf_mill = fz * z * n_mill
Q_mill  = ap_mill * ae * vf_mill / 1000
kc_mill = kc1 * fz ** (-mc)
Pc_mill = ap_mill * ae * vf_mill * kc_mill / 60e6

print(f"n = {n_mill:.0f} rpm, table feed = {vf_mill:.0f} mm/min")
print(f"Q = {Q_mill:.0f} cm^3/min, Pc = {Pc_mill:.1f} kW")

## The same ideas in drilling

A drill makes a round hole of area $\pi D^2/4$ and advances $v_f = f_n \cdot n$ millimetres per minute. Removal rate and power become:

$$Q = \frac{v_c \cdot D \cdot f_n}{4} \qquad P_c = \frac{f_n \cdot v_c \cdot D \cdot k_c}{240 \cdot 10^3}$$

A drill has two cutting edges sharing the feed, so each chip is about half the feed thick.

In [ ]:
D_drill  = 12    # mm
vc_drill = 100   # m/min
fn_drill = 0.2   # mm/rev
depth    = 40    # mm

n_drill  = vc_drill * 1000 / (math.pi * D_drill)
vf_drill = fn_drill * n_drill
h_drill  = fn_drill / 2 * math.sin(math.radians(70))   # 140 degree point angle
kc_drill = kc1 * h_drill ** (-mc)
Pc_drill = fn_drill * vc_drill * D_drill * kc_drill / 240e3

print(f"n = {n_drill:.0f} rpm, feed = {vf_drill:.0f} mm/min, time = {depth / vf_drill * 60:.1f} s")
print(f"Pc = {Pc_drill:.2f} kW")

## Try it in class

1. The bar is now 40 mm in diameter. What rpm do you need for $v_c = 200$ m/min? Is that what you expected?
2. Using `turning_job`, find the largest feed $f_n$ that stays under 8 kW of cutting power at $v_c = 200$ m/min and $a_p = 3$ mm.
3. Aluminium has roughly $k_{c1} = 500$ N/mm². How much less power does the same job need in aluminium? (Pass `kc1=500` to `turning_job`.)

In [ ]:
# your work here